<a href="https://colab.research.google.com/github/NeoByteVoyager/DeepLearning/blob/main/HW/CNN/MNIST_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 准备数据

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

# 数据转换
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 下载数据集
train_dataset = torchvision.datasets.MNIST(
    root="./dataset",
    train=True,
    transform=transform,
    download=True
)

test_dataset = torchvision.datasets.MNIST(
    root="./dataset",
    train=False,
    transform=transform,
    download=True
)

# 加载数据集
train_load = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_load = DataLoader(dataset=test_dataset,batch_size=64,shuffle=False)


cuda
epoch:0|loss:0.3096
准确率:0.9642%
epoch:1|loss:0.1053
准确率:0.9779%
epoch:2|loss:0.07773
准确率:0.9829%


## 搭建模型

In [ ]:
# 搭建模型
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=64, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=64, out_channels=32, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Flatten(),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        return self.network(x)

# 实例化模型
model = MyModel()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

## 损失函数和优化器

In [ ]:
# 创建损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr = 0.01)

## 训练循环

In [ ]:
def train():
    total_loss = 0

    for features, labels in train_load:
        features = features.to(device)
        labels = labels.to(device)
        y_hat = model(features)
        loss = criterion(y_hat,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()   # 这里的loss算的是一个批次的loss

    return total_loss / len(train_load)

def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for features, labels in test_load:
            features = features.to(device)
            labels = labels.to(device)
            y_hat = model(features)

            _, preds =  torch.max(y_hat, dim=1)

            correct += (preds==labels).sum().item()
            total +=features.shape[0]
    print(f"准确率:{correct/total}%")

for epoch in range(10):
    loss = train()
    print(f"epoch:{epoch}|loss:{loss:.4}")
    test()

